# 2 — Retrieval and Generation

Now we ask questions of the index we built in notebook 1.

Four things happen, and all four are used in real systems:

```
search  →  rerank  →  filter  →  fetch table rows  →  answer
```

The code lives in the `rag` package, same as notebook 1. Each cell here is a few
lines — enough to see what a step does. Open `rag/retrieval.py` if you want the
details.

| § | What |
|---|---|
| 0 | Setup |
| 1 | Search and rerank |
| 2 | Filtering with metadata |
| 3 | Table rows |
| 4 | Build the prompt and answer |

---
## 0. Setup

In [17]:
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
import os

os.environ.setdefault("OPENAI_API_KEY", "")
os.environ.setdefault("PINECONE_API_KEY", "")

'pcsk_51ZQWd_5YGgauDgUMCf26y1xHSrSBnbAgdVMqP9JEUNU34Ry4mXzZ82kfqRjnUSMsSSbnD'

In [19]:
from dotenv import load_dotenv

# Read environment variables from the .env file, if one exists.
load_dotenv()

True

In [20]:
from dotenv import load_dotenv
from pathlib import Path
import os
print("OpenAI key loaded:", bool(os.getenv("OPENAI_API_KEY")))
print("Pinecone key loaded:", bool(os.getenv("PINECONE_API_KEY")))

OpenAI key loaded: True
Pinecone key loaded: True


In [21]:
from rag.index import open_index
from rag.retrieval import search

index = open_index()

query = "Can you describe how many types of leaves available?"

results = search(
    index,
    query,
    top_k=5
)

print("Number of results:", len(results))

for i, result in enumerate(results, 1):
    print("\n" + "=" * 80)
    print(f"RESULT {i}")
    print("=" * 80)

    print("Chunk ID :", result["chunk_id"])
    print("Page     :", result["page"])
    print("Type     :", result["content_type"])
    print("Dense    :", result["dense"])
    print("Rerank   :", result["rerank"])
    print("\nText:")
    print(result["text"][:1000])

Number of results: 5

RESULT 1
Chunk ID : leavepolicy:6189ce30280f671b:0
Page     : 4
Type     : text
Dense    : 0.3573
Rerank   : 0.6853

Text:
Types of leaves:
6. Paternity leave :
Paternity leave is unpaid leave given to a male employee hen a child is born, duration of the leave is at the sole discretion of the management.

RESULT 2
Chunk ID : leavepolicy:c9c3439cdc73bd4e:0
Page     : 4
Type     : text
Dense    : 0.3602
Rerank   : 0.6351

Text:
Types of leaves:
7. Leave without pay :
7. Leave ithout pay :
If person does not have any unused leave and the situation ants him to take the leave, the leave is granted by the Company as loss of pay or hich may be adjusted against the future leave or as a special case depending on management discretion.
Casual leave cannot be clubbed ith any other leave but Sick Leave and Privilege leave can be clubbed together.
Arpit Agar al Director Date:1 st April 2022

RESULT 3
Chunk ID : leavepolicy:f289f40c3351d859:0
Page     : 3
Type     : text
Dense 

Two score columns. `dense` is the vector distance. `rerank` is the cross-encoder
reading your question and the chunk together.

Where they disagree is where the second stage earned its cost.